In [2]:
import pandas as pd
import numpy as np
#from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.preprocessing.sequence import pad_sequences

# revised methodology: avoids leakage from same group over time

# 1. load data
# reading in resulting CSV from aggregate_and_build.py script 
# data = pd.read_csv("lstm_preprocessed_data.csv")
data = pd.read_csv("processed/all_years_aggregated.csv")


2026-04-19 18:02:37.237126: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:

# 2. group id
data["group_id"] = (
    data["geo_region"].astype(str) + "_" +
    data["education_grouped"].astype(str) + "_" +
    data["age_group"].astype(str) + "_" +
    data["family_income_grouped"].astype(str)
)

# 3. survival filter
group_survival = (
    data.groupby("group_id")["year"]
    .nunique()
    .reset_index(name='year_count')
)

# making sure groups are consistent between all 8 years 
# true time series will track same groups over time 
surviving_groups = group_survival[group_survival["year_count"] >= 8]["group_id"]

model_data = data[data["group_id"].isin(surviving_groups)].copy()

# 4. sort
model_data = model_data.sort_values(["group_id", "year"])

# 5. fill missing
model_data = model_data.fillna(0)

In [4]:
# remove columns we already grouped on - don't need them as features! - redundant
# also werent properly encoded so were causing typecast errors 
model_data = model_data.drop(columns=["age_group", "education_grouped","family_income_grouped"])

# 6. features
# everything except the group id, year, and target variable 
feature_cols = [
    c for c in model_data.columns
    if c not in ["group_id", "year", "did_vote_1"]
]
# 7. buidling sequences for each group
# need to filter for years then build sequences after
train_years = 2018
val_years   = 2022
test_years  = 2024
def build_sequences(df, max_year):
    X_seq, y_seq = [], []

    for gid, g in df.groupby("group_id"):
        g = g.sort_values("year")

        g = g[g["year"] <= max_year]   # filter to find rows in corrct year

        if len(g) < 2:
            continue

        X_seq.append(g[feature_cols].values) # only rows with correct year added to sequence
        y_seq.append(g["did_vote_1"].values) # only rows with coreect year added to sequence

    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)


X_train, y_train = build_sequences(model_data, 2018)
X_val, y_val     = build_sequences(model_data, 2022)
X_test, y_test   = build_sequences(model_data, 2024)



'''
X_seq, y_seq = [], []

for gid, g in model_data.groupby("group_id"):
    g = g.sort_values("year")

    X_seq.append(g[feature_cols].values) # creating a sequence of features per group 
    y_seq.append(g["did_vote_1"].values) # creating a sequence of target feature value per group
# each group now has a sequence of X and y values  

# 8. padding
# add zeros at end if uneven time series lengths between groups
# SHOULDN"T NEED IF WE ALREADY HAVE USED THE GROUP SURVIVAL 
# already ensured same 8 years are non-missing for each group 
# X = pad_sequences(X_seq, dtype="float32", padding="post") 
# y = pad_sequences(y_seq, dtype="float32", padding="post")
X = np.array(X_seq)
y = np.array(y_seq)

'''
''''
# This is group-level splitting logic 
# each group would only be in training, testing, or validation set
# map from group to index 
group_ids = list(model_data["group_id"].unique())
group_to_idx = {g:i for i, g in enumerate(group_ids)}

# split sequences by group identity
# not by rows
train_groups = model_data[model_data["year"] <= 2018]["group_id"].unique()
val_groups   = model_data[(model_data["year"] > 2018) & (model_data["year"] <= 2022)]["group_id"].unique()
test_groups  = model_data[model_data["year"] == 2024]["group_id"].unique()

# group IDs to indices
train_idx = [group_to_idx[g] for g in train_groups if g in group_to_idx]
val_idx   = [group_to_idx[g] for g in val_groups if g in group_to_idx]
test_idx  = [group_to_idx[g] for g in test_groups if g in group_to_idx]

# slice sequence arrays 
# avoid temporal data leakage 
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val     = X[val_idx], y[val_idx]
X_test, y_test   = X[test_idx], y[test_idx]

'''
'''
# THIS IS INVALID BECAUSE HAS LEAKAGE
# split into training and testing 
# this is year level split
# each year snapshot would either be in train, test, or validation set
# train on years <- 2018, validate on 2020-2022, test on 2024
train_data = model_data[model_data["year"] <= 18] # for training
val_data   = model_data[(model_data["year"] > 18) & (model_data["year"] <= 22)] # for tuning/selection
# after final model has been tuned/optimized, avoid data leakage
test_data  = model_data[model_data["year"] == 24] # for testing performance on unseen observations 
'''

'\n# THIS IS INVALID BECAUSE HAS LEAKAGE\n# split into training and testing \n# this is year level split\n# each year snapshot would either be in train, test, or validation set\n# train on years <- 2018, validate on 2020-2022, test on 2024\ntrain_data = model_data[model_data["year"] <= 18] # for training\nval_data   = model_data[(model_data["year"] > 18) & (model_data["year"] <= 22)] # for tuning/selection\n# after final model has been tuned/optimized, avoid data leakage\ntest_data  = model_data[model_data["year"] == 24] # for testing performance on unseen observations \n'

In [5]:
print(X_train.shape)


(1620, 8, 59)


In [7]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

# Define input parameters
n_timesteps = X_train.shape[1]
n_features = X_train.shape[2]
# set up the architecture - subject to change
model = Sequential([
    # Input layer with input_shape matching your sequences
    LSTM(64, activation='relu', input_shape=(n_timesteps, n_features), return_sequences=False),
    
    # Optional: Add Dropout to prevent overfitting
    Dropout(0.2),
    
    # Output layer: 1 for single-value prediction, or len(y_train[0]) for multi-step
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')



/Users/ellawileman/anaconda3/envs/ds_env/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [8]:
type(X_train)

numpy.ndarray

In [9]:
type(y_train)

numpy.ndarray

In [10]:
X_train.shape

(1620, 8, 59)

In [11]:
print(X_train.dtype)
print(type(X_train[0][0][0]))

float32
<class 'numpy.float32'>


In [12]:
print(X_train.dtype)
print(type(X_train[0]))

float32
<class 'numpy.ndarray'>


In [13]:
# Train the model
# history = model.fit(X_train, y_train, epochs=20, batch_size=256, verbose=1)

In [ ]:
from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=5,           # Wait 5 epochs for improvement
    restore_best_weights=True # Revert to the best model
)

# Use in model.fit
model.fit(
    X_train, y_train, 
    epochs=100,           # Set a high max, let early stopping take over
    validation_data=(X_val, y_val),
    callbacks=[early_stop], 
    verbose = 1
)


Epoch 1/100
